# Efficient Analysis — Projection Geometry First

Ordered to minimise generation cost:

| Phase | What it does | Generation? |
|-------|--------------|-------------|
| **1. Existing data** | SVD + validation metrics already in manifest | None |
| **2. Gate: step-count validation** | Does variance trajectory shape survive at 4–8 steps? | ~100 images |
| **3+. Deferred stubs** | Described in markdown; coded once gate passes | TBD |

Texture-coloring (Exp 8 corpus swap) is the lowest-priority item and is **not coded here**.
It will be added after all other experiments have produced data to compare against,
to ensure the corpus retrain does not invalidate prior findings.

---
**Codebase prerequisite (already applied):**  
`experiment.py:461` now reads `num_inference_steps` from the config JSON (default 30).
Pass `"num_inference_steps": 8` in a config to get a fast survey pass.

## 1 · Setup

In [ ]:
import os, sys, json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

IN_COLAB = 'google.colab' in sys.modules or bool(os.environ.get('COLAB_RELEASE_TAG'))

REPO_URL    = 'https://github.com/leonorae/slicer'
REPO_BRANCH = 'claude/analyze-experiment-confounders-uBu9h'

if IN_COLAB:
    REPO_DIR    = Path('/content/slicer')
    DRIVE_BASE  = Path('/content/drive/MyDrive/diffusion_microscope')
    RESULTS_DIR = DRIVE_BASE / 'experiment_results_efficient'
    HF_CACHE    = DRIVE_BASE / 'hf_cache'
else:
    REPO_DIR    = Path('/home/user/slicer')
    DRIVE_BASE  = None
    RESULTS_DIR = REPO_DIR / 'experiment_results'
    HF_CACHE    = None

# Existing results directory — where the trained manifest already lives.
# Override this if your results are elsewhere.
EXISTING_RESULTS = RESULTS_DIR

print(f'IN_COLAB       : {IN_COLAB}')
print(f'REPO_DIR       : {REPO_DIR}')
print(f'EXISTING_RESULTS: {EXISTING_RESULTS}')

In [ ]:
# Colab only — mount Drive, clone repo, install deps
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    import subprocess
    print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout or 'No GPU')

    if REPO_DIR.is_dir():
        !git -C {REPO_DIR} fetch origin {REPO_BRANCH}
        !git -C {REPO_DIR} checkout {REPO_BRANCH}
        !git -C {REPO_DIR} reset --hard origin/{REPO_BRANCH}
    else:
        !git clone --branch {REPO_BRANCH} --single-branch {REPO_URL} {REPO_DIR}

    !pip install -q open-clip-torch diffusers Pillow lpips datasets nltk sentencepiece accelerate scikit-learn
    !pip install -q -e {REPO_DIR} --no-deps

    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    HF_CACHE.mkdir(parents=True, exist_ok=True)
    os.environ['HF_HOME'] = str(HF_CACHE)
    os.environ['TRANSFORMERS_CACHE'] = str(HF_CACHE)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

---
## Phase 1 — Existing projection geometry (no generation)

The manifest contains per-layer SVD and validation metrics for whatever alpha values
were trained, computed during the training phase.
No images, no LLM re-runs, no SD — pure manifold geometry.

**What we can learn here:**
- How erank and condition number vary with layer depth and regularisation strength
- Where the R² ↔ nn_recall@5 trade-off is sharpest (those layers are most sensitive to alpha choice)
- Which alpha values are worth running in the gating experiment (skip those that collapse or are numerically identical)

In [ ]:
manifest_path = EXISTING_RESULTS / 'manifest.json'
if not manifest_path.exists():
    print(f'No manifest found at {manifest_path}')
    print('If this is a fresh run, point EXISTING_RESULTS at a directory with a completed train phase.')
    manifest = {}
else:
    with open(manifest_path) as f:
        manifest = json.load(f)

projections = manifest.get('projections', {})
print('Projection keys:', list(projections.keys()))
print('Manifest top-level keys:', [k for k in manifest if k != 'projections'])

In [ ]:
# Discover projection keys from the manifest — works for any alpha set.
import re as _re

def _parse_alpha(pk):
    """Extract alpha label from a proj_key like per_layer_alpha1000 or per_layer_alphaauto."""
    m = _re.search(r'alpha(.+)$', pk)
    if not m:
        return pk
    raw = m.group(1)
    try:
        v = float(raw)
        return int(v) if v == int(v) else v
    except ValueError:
        return raw   # 'auto' etc.

# Build PROJ_KEYS from whatever is in this manifest
PROJ_KEYS = sorted(projections.keys())

if not PROJ_KEYS:
    print('No projection keys found in manifest — was the train phase run?')
else:
    print(f'Found {len(PROJ_KEYS)} projection(s): {PROJ_KEYS}')

# Assign colours by position so any number of alphas renders cleanly
_PALETTE = ['#E53935', '#FB8C00', '#FDD835', '#43A047', '#1E88E5',
            '#8E24AA', '#00ACC1', '#6D4C41']
ALPHA_COLORS = {pk: _PALETTE[i % len(_PALETTE)] for i, pk in enumerate(PROJ_KEYS)}
ALPHA_LABELS = {pk: f'α={_parse_alpha(pk)}' for pk in PROJ_KEYS}


def get_metric(proj_key, metric_group, metric_name):
    """Return list of (layer_idx, value) sorted by layer, skipping +inf (singular)."""
    proj  = projections.get(proj_key, {})
    group = proj.get(metric_group, {})
    pairs = []
    for layer_str, vals in group.items():
        try:
            layer = int(layer_str)
            v = vals.get(metric_name)
            if v is not None and not (isinstance(v, float) and (np.isinf(v) or np.isnan(v))):
                pairs.append((layer, v))
        except (ValueError, AttributeError, TypeError):
            pass
    return sorted(pairs)


# Sanity table
print(f'\n{"proj_key":35s}  {"L0 erank":>8}  {"Lmax erank":>10}  {"L0 cond":>10}  {"L0 R²":>6}  {"Lmax nn@5":>9}')
for pk in PROJ_KEYS:
    erank = dict(get_metric(pk, 'svd', 'erank'))
    cond  = dict(get_metric(pk, 'svd', 'condition_number'))
    r2    = dict(get_metric(pk, 'validation', 'projection_r2'))
    nnr   = dict(get_metric(pk, 'validation', 'nn_recall_at_5'))
    if not erank:
        print(f'{pk:35s}  (no SVD data)')
        continue
    L0   = min(erank)
    Lmax = max(erank)
    cond_v = cond.get(L0)
    cond_s = f'{cond_v:.2e}' if cond_v is not None else '       ?'
    print(f'{pk:35s}  {erank.get(L0, 0):8.0f}  {erank.get(Lmax, 0):10.0f}  '
          f'{cond_s:>10}  '
          f'{r2.get(L0, float("nan")):6.3f}  '
          f'{nnr.get(Lmax, float("nan")):9.3f}')

In [ ]:
# ── SVD trajectories: erank, condition_number, n_visible ──────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

SVD_METRICS = [
    ('erank',            'Effective rank (erank)',        False),
    ('condition_number', 'Condition number (log scale)',  True),
    ('n_visible',        'n_visible (non-zero SV dims)',  False),
]

for ax, (metric, ylabel, log_y) in zip(axes, SVD_METRICS):
    for pk in PROJ_KEYS:
        pairs = get_metric(pk, 'svd', metric)
        if not pairs:
            continue
        xs, ys = zip(*pairs)
        ax.plot(xs, ys, marker='o', ms=3, lw=1.5,
                color=ALPHA_COLORS[pk], label=ALPHA_LABELS[pk])
    ax.set_xlabel('Layer')
    ax.set_ylabel(ylabel)
    ax.set_title(ylabel)
    if log_y:
        ax.set_yscale('log')
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.25)

plt.suptitle('Projection geometry — computed from Ridge W matrix alone (no generation)', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# ── Validation metrics: R² and nn_recall@5 per layer, per alpha ───────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, (metric, ylabel) in zip(axes, [
    ('projection_r2',   'R² (validation corpus)'),
    ('nn_recall_at_5',  'nn_recall@5 (topology preservation)'),
]):
    for pk in PROJ_KEYS:
        pairs = get_metric(pk, 'validation', metric)
        if not pairs:
            continue
        xs, ys = zip(*pairs)
        ax.plot(xs, ys, marker='o', ms=3, lw=1.5,
                color=ALPHA_COLORS[pk], label=ALPHA_LABELS[pk])
    ax.set_xlabel('Layer')
    ax.set_ylabel(ylabel)
    ax.set_title(ylabel)
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.25)

plt.suptitle('R² vs nn_recall@5 — these pull in opposite directions; high alpha maximises R² but kills topology',
             fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# ── R² vs nn_recall scatter: one point per (alpha, layer), coloured by alpha ──
# Shows the trade-off frontier. The "interesting" regime is the upper-right corner
# (high nn_recall AND high R²), which is probably unreachable.
fig, ax = plt.subplots(figsize=(7, 5))

for pk in PROJ_KEYS:
    r2_dict  = dict(get_metric(pk, 'validation', 'projection_r2'))
    nnr_dict = dict(get_metric(pk, 'validation', 'nn_recall_at_5'))
    common   = sorted(set(r2_dict) & set(nnr_dict))
    if not common:
        continue
    xs = [r2_dict[l]  for l in common]
    ys = [nnr_dict[l] for l in common]
    sc = ax.scatter(xs, ys, c=[l for l in common], cmap='plasma',
                    s=40, alpha=0.8, label=ALPHA_LABELS[pk],
                    edgecolors=ALPHA_COLORS[pk], linewidths=1.5)

ax.set_xlabel('R² (reconstruction fidelity)')
ax.set_ylabel('nn_recall@5 (topology preservation)')
ax.set_title('R² vs nn_recall@5 trade-off front\nMarker depth = layer (bright = deep layers)')
ax.legend(fontsize=7, loc='lower right')
ax.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()

# Print the alpha values with best nn_recall at the last layer
last_layer = max(
    int(l) for pk in PROJ_KEYS for l, _ in get_metric(pk, 'validation', 'nn_recall_at_5')
) if any(get_metric(pk, 'validation', 'nn_recall_at_5') for pk in PROJ_KEYS) else None

if last_layer is not None:
    print(f'\nnn_recall@5 at last layer (L{last_layer}) by alpha:')
    for pk in PROJ_KEYS:
        d = dict(get_metric(pk, 'validation', 'nn_recall_at_5'))
        v = d.get(last_layer, float('nan'))
        print(f'  {ALPHA_LABELS[pk]:20s}  nn_recall@5 = {v:.4f}')

In [ ]:
# ── Summarise which alphas are present and what they look like ────────────────
print('Alpha summary at last layer:')
print()

last_layer = None
for pk in PROJ_KEYS:
    layers = [l for l, _ in get_metric(pk, 'validation', 'nn_recall_at_5')]
    if layers:
        last_layer = max(last_layer or 0, max(layers))

rows = []
for pk in PROJ_KEYS:
    erank  = dict(get_metric(pk, 'svd', 'erank'))
    nnr    = dict(get_metric(pk, 'validation', 'nn_recall_at_5'))
    layers = sorted(set(erank) & set(nnr))
    if not layers:
        continue
    last = max(layers)
    rows.append((pk, erank.get(min(layers)), erank.get(last), nnr.get(last, float('nan'))))
    print(f'  {ALPHA_LABELS[pk]:20s}  '
          f'erank(L0)={erank.get(min(layers), "?"):>5.0f}  '
          f'erank(L{last})={erank.get(last, "?"):>5.0f}  '
          f'nn_recall(L{last})={nnr.get(last, float("nan")):.3f}')

print()

# Dynamic recommendation: identify lowest-alpha and highest-alpha keys present
if rows:
    best_nnr = max(rows, key=lambda r: r[3])
    worst_nnr = min(rows, key=lambda r: r[3])
    print(f'Best nn_recall@5:   {ALPHA_LABELS[best_nnr[0]]}  (use for gating — most discriminative images)')
    print(f'Worst nn_recall@5:  {ALPHA_LABELS[worst_nnr[0]]}  (corpus prior regime — useful as compression baseline)')
    if best_nnr[0] != worst_nnr[0]:
        print()
        print(f'Recommendation: run gating experiment with at minimum '
              f'{ALPHA_LABELS[best_nnr[0]]} and {ALPHA_LABELS[worst_nnr[0]]}.')
    if len(rows) > 2:
        print(f'Intermediate alpha values ({", ".join(ALPHA_LABELS[r[0]] for r in rows[1:-1])}) '
              f'can be added after gate passes if finer granularity is needed.')

---
## Phase 2 — Gate: does step-count reduction preserve the variance trajectory shape?

**Goal:** confirm that 4–8 denoising steps produces the same *ordering* of layer variance
as 30 steps, even if absolute values differ.  If yes, future survey passes can use
4–8 steps and be ~4–8× cheaper.

**Design:** 1 probe × all 24 layers × 4 seeds × 4 step counts `[4, 8, 20, 30]`.  
~100–400 images depending on how many step counts you run in parallel.

**Decision rule:**  
Compute Spearman rank correlation of mean_pixel_var(layer) between each step count
and the full 30-step baseline.  If ρ ≥ 0.85 for 8 steps, 8-step survey passes are valid.

**Prerequisite:** a trained projection must exist (train phase completed).
The cell below checks for this and skips if weights are missing.

In [ ]:
import subprocess

GATE_RESULTS = (DRIVE_BASE if IN_COLAB else REPO_DIR) / 'experiment_results_gate'
GATE_RESULTS.mkdir(parents=True, exist_ok=True)

GATE_PROBE = 'a cat'

GATE_STEP_COUNTS = [4, 8, 20, 30]

# Pick the alpha with the best nn_recall from whatever was trained.
# Falls back to 1 if we can't determine it (e.g. manifest is empty).
if rows:
    GATE_ALPHA = _parse_alpha(best_nnr[0])
    # Ensure it's a number the config can accept (auto → use 1 as fallback)
    if not isinstance(GATE_ALPHA, (int, float)):
        GATE_ALPHA = 1
else:
    GATE_ALPHA = 1

print(f'Gate alpha: {GATE_ALPHA}  (best nn_recall from trained projections)')

# Check projection weights exist
proj_dir = EXISTING_RESULTS / 'projections'
if not proj_dir.exists():
    print(f'\nWARNING: no projection weights found at {proj_dir}')
    print('The gating experiment needs a completed train phase.')
    print('Run: python run_experiment.py --config <your_config> --phase train')
    GATE_READY = False
else:
    GATE_READY = True
    print(f'Projection weights found — ready to run gating experiment.')
    print(f'Probe: "{GATE_PROBE}"  |  Step counts: {GATE_STEP_COUNTS}  |  Output: {GATE_RESULTS}')

In [ ]:
# Write one config per step count.
# Each shares the trained projection from EXISTING_RESULTS by pointing output there;
# images go into a gating-specific subdir to keep results clean.

if not GATE_READY:
    print('Skipping — projection weights not found.')
else:
    gate_cfg_paths = {}

    # Load base config from existing results to get model IDs etc.
    base_cfg = manifest.get('config', {})

    for steps in GATE_STEP_COUNTS:
        out_dir = GATE_RESULTS / f'steps_{steps}'
        out_dir.mkdir(exist_ok=True)

        cfg = {
            '_comment': f'Gate experiment: step-count validation at {steps} steps.',
            'models': base_cfg.get('models', {
                'llm': 'EleutherAI/pythia-410m',
                'sd':  'sd-legacy/stable-diffusion-v1-5',
                'clip_model': 'ViT-L-14',
                'clip_pretrained': 'openai',
            }),
            'projections': {
                'types': ['per_layer'],
                'alpha_values': [GATE_ALPHA],
                # training_data_size irrelevant — we skip train phase
                'training_data_size': 100,
            },
            'probe_texts': {'gate': [GATE_PROBE]},
            'layers': list(range(24)),   # all 24 layers
            'cfg_values': [25.0],
            'seeds': [42, 123, 777, 456],  # 4 seeds for reliable seed variance
            'num_inference_steps': steps,
            'track_lpips': False,
            'output': {
                'base_dir': str(out_dir),
                'image_format': 'png',
            },
        }

        cfg_path = REPO_DIR / f'_gate_config_steps{steps}.json'
        cfg_path.write_text(json.dumps(cfg, indent=2))
        gate_cfg_paths[steps] = cfg_path
        print(f'Wrote {cfg_path}')

    print()
    print('NOTE: the generate phase will try to train first if no manifest exists.')
    print('Copy the trained projection weights from EXISTING_RESULTS/projections/')
    print('into each GATE_RESULTS/steps_N/ dir before running generate, or run')
    print('train (cheap at 100 samples) and then generate.')

In [ ]:
%%time
# Run gate experiment — train + generate for each step count.
# Train is fast (100 samples, 1 alpha, 24 layers).
# Generate: 1 probe × 24 layers × 4 seeds = 96 images per step count.
# Total: ~384 images across 4 step counts.

if not GATE_READY:
    print('Skipping — projection weights not found.')
else:
    for steps, cfg_path in gate_cfg_paths.items():
        print(f'\n── {steps} steps ──────────────────────────────────────────')
        for phase in ['train', 'generate']:
            result = subprocess.run(
                ['python', str(REPO_DIR / 'run_experiment.py'),
                 '--config', str(cfg_path),
                 '--phase', phase],
                capture_output=True, text=True, cwd=str(REPO_DIR)
            )
            if result.returncode != 0:
                print(f'FAILED ({phase}):')
                print(result.stderr[-2000:])
            else:
                last = result.stdout.strip().splitlines()[-3:]
                print(f'  {phase}: OK — {chr(10).join(last)}')

In [ ]:
from scipy.stats import spearmanr

# Load seed_variance from each step-count manifest
# and build layer → mean_pixel_var for the gate probe.

def gate_slug(text):
    """Replicate experiment.py _slug() — lowercase, underscores, truncated."""
    import re
    s = re.sub(r'[^a-z0-9]+', '_', text.lower()).strip('_')
    return s[:50]

GATE_SLUG = gate_slug(GATE_PROBE)
gate_trajectories = {}  # steps → {layer: mean_pixel_var}

for steps in GATE_STEP_COUNTS:
    mpath = GATE_RESULTS / f'steps_{steps}' / 'manifest.json'
    if not mpath.exists():
        print(f'steps={steps}: manifest not found (run gate-run cell first)')
        continue
    with open(mpath) as f:
        m = json.load(f)
    sv = m.get('seed_variance', {})
    traj = {}
    for key, rec in sv.items():
        # key: "{proj_key}/{slug}/L{NNNN}/CFG{v}"
        parts = key.split('/')
        if len(parts) < 4:
            continue
        slug    = parts[-3]
        layer_s = parts[-2]
        if slug != GATE_SLUG or not layer_s.startswith('L'):
            continue
        layer = int(layer_s[1:])
        traj[layer] = rec.get('mean_pixel_var')
    gate_trajectories[steps] = traj
    print(f'steps={steps:2d}: {len(traj)} layers loaded, '
          f'min_var={min(traj.values()):.0f} at L{min(traj, key=traj.get)}')

if not gate_trajectories:
    print('No gate data yet — run the gating experiment cells above first.')

In [ ]:
if not gate_trajectories:
    print('No gate data to plot.')
else:
    step_colors = {4: '#EF5350', 8: '#FFA726', 20: '#66BB6A', 30: '#42A5F5'}
    baseline = gate_trajectories.get(30, {})

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

    # Left: raw trajectories
    for steps, traj in sorted(gate_trajectories.items()):
        xs = sorted(traj)
        ys = [traj[x] for x in xs]
        ax1.plot(xs, ys, marker='o', ms=3, lw=1.5,
                 color=step_colors.get(steps, '#aaa'),
                 label=f'{steps} steps')
    ax1.set_xlabel('Layer')
    ax1.set_ylabel('Mean pixel variance (across seeds)')
    ax1.set_title(f'Gate: variance trajectory vs step count\n"{GATE_PROBE}"')
    ax1.legend()
    ax1.grid(True, alpha=0.25)

    # Right: Spearman ρ vs 30-step baseline
    if baseline:
        rhos, labels_rho = [], []
        for steps in sorted(gate_trajectories):
            if steps == 30:
                continue
            traj = gate_trajectories[steps]
            common = sorted(set(traj) & set(baseline))
            if len(common) < 3:
                continue
            ys_test = [traj[l]     for l in common]
            ys_base = [baseline[l] for l in common]
            rho, pval = spearmanr(ys_test, ys_base)
            rhos.append(rho)
            labels_rho.append(f'{steps} steps\nρ={rho:.3f}')
            print(f'{steps:2d} steps vs 30: Spearman ρ={rho:.3f}, p={pval:.3f}')

        colors_rho = [step_colors.get(int(l.split()[0]), '#aaa') for l in labels_rho]
        bars = ax2.bar(labels_rho, rhos, color=colors_rho, alpha=0.85, edgecolor='white')
        ax2.axhline(0.85, linestyle='--', color='white', alpha=0.6,
                    label='ρ=0.85 threshold')
        ax2.set_ylim(0, 1.05)
        ax2.set_ylabel('Spearman ρ vs 30-step baseline')
        ax2.set_title('Shape preservation — ρ ≥ 0.85 → safe for survey passes')
        ax2.legend(fontsize=8)
        ax2.grid(True, alpha=0.25, axis='y')

    plt.tight_layout()
    plt.show()

    # Gate decision
    print()
    for steps in sorted(gate_trajectories):
        if steps == 30 or steps not in gate_trajectories:
            continue
        traj = gate_trajectories[steps]
        common = sorted(set(traj) & set(baseline))
        if len(common) < 3:
            continue
        rho, _ = spearmanr([traj[l] for l in common], [baseline[l] for l in common])
        status = 'PASS ✓' if rho >= 0.85 else 'FAIL ✗'
        print(f'{steps:2d} steps: ρ={rho:.3f}  →  {status}')
    print()
    print('If 8 steps passes: use num_inference_steps=8 for all survey passes.')
    print('If only 20 steps passes: use 20 for surveys (still ~1.5× speedup).')
    print('If nothing passes: step-count reduction is not safe; use 30 steps throughout.')

---
## Phase 3+ — Deferred: coded after gating passes

The cells below are **not yet written**. They will be added once the gating experiment
confirms the appropriate step count for survey passes.
Each section describes what it needs and what it produces.

---

### 3a · Survey pass — Exp 4 full layer sweep (Pythia-410m, 9 probes, 24 layers)

**Needs:** gate passed, `num_inference_steps` validated, projection weights from train phase.  
**What:** 1–2 seeds × 24 layers × 9 probes × α=1 at the validated survey step count.  
**Produces:** `seed_variance` in manifest for all (probe, layer) pairs — cheap proxy trajectory.  
**Next step:** use CLIP drift (from `probe_clip_vectors`) to identify interesting layers;
run full 16-seed pass only at convergence ± 2 layers.

---

### 3b · CLIP drift per layer (zero generation)

**Needs:** `probe_clip_vectors` populated in manifest (from any generate phase).  
**What:** `cosine_distance(proj_Ln, proj_L{n-1})` per (probe, layer) — 23-dim drift vector.  
**Produces:** transition layer map — high drift = boundary between representational regimes.  
**Used for:** scheduling which layers get full 16-seed treatment.

---

### 3c · Attractor structure (Exp 7) — from existing seeds

**Needs:** ≥ 8 seed images per (probe, layer) from any completed generate phase.  
**What:** k-means (k=1–4) on 16 seeds per (probe, layer) using pixel-distance or LPIPS.  
**Produces:** `attractor_count` per (probe, layer); within-cluster mean images; between-cluster LPIPS.  
**Insight:** distinguishes unimodal convergence layers from genuinely bimodal layers
(different visual content, not just noise) from diffuse prior-dominated layers
(uniform noise, single wide cluster).

---

### 3d · Data-driven probe categories (Exp 9) — trajectory fingerprinting

**Needs:** `seed_variance` populated for all 24 layers for ≥ 10 probes.  
**What:** cluster probes by their 24-dim variance trajectory shape (z-scored, DTW or cosine);
also cluster by the 24-dim compression-sensitivity trajectory (cosine dist α=1 vs α=1000 per layer).  
**Produces:** data-driven probe clusters; post-hoc overlay of Exp 5 corpus categories.
PCA of the trajectory matrix — first two PCs describe dominant variation modes.  
**Insight:** categories that emerge from representational geometry without reference
to corpus provenance or content assumptions.

---

### 4 · Texture-coloring / corpus-stability test (Exp 8) — lowest priority

**Needs:** Exp 4 full layer sweep results (survey + full pass) and trajectory fingerprinting
results from 3d as baseline. Requires re-running train phase on a Wikipedia-heavy corpus.
This corpus retrain changes both W (the projection matrix) and b (the centroid/bias),
so it potentially invalidates all prior projection comparisons if run first.  
**Will be added only after** all other experiments have produced stable results
to compare against.

**What it tests:** does the convergence layer index (minimum mean_pixel_var per probe)
shift when corpus changes from image-caption-heavy to Wikipedia-heavy?  
- **If stable:** convergence layer is an architecture property, not a corpus artifact.  
- **If shifts:** corpus is doing more representational work than expected.  
**Secondary:** visual texture at the convergence layer shifts (different centroid aesthetic)
without the convergence layer index moving — confirms the corpus acts as an aesthetic
knob on the *content* at the convergence point, not on the *location* of it.